<h4>Astreoids Risk Analysis Dashboard (ARAD) Risk Model Notebook<h4> 
<p>This notebook will be used to test a Support Vector Classification (SVC) model.</p>

In [1]:
import os 
import sqlite3 
import pandas as pd 

In [2]:
db_path = os.path.join("..","astreoid_risk_anaylzed.db")
conn = sqlite3.connect(db_path) 
df = pd.read_sql("SELECT * FROM astreoids WHERE close_date",conn) 

In [3]:
df.head(5)

,close_date,astreoid_name,max_diameter,min_diameter,absoulte_magnitude,min_distance_km,min_distance_au,speed_kmh,potentially_hazardous,sentry_object
0,2026-05-24,(2010 SX11),0.065169,0.029144,24.80,6.167525e+07,0.412274,26751.080414,0,0
1,2026-05-23,(2000 UR16),0.098637,0.044112,23.90,7.445741e+07,0.497717,74366.251824,0,0
2,2026-05-22,(2011 UD256),0.622358,0.278327,19.90,6.209497e+07,0.415079,128183.818181,0,0
3,2026-05-21,136818 Selqet (1997 MW1),0.950692,0.425163,18.98,2.004144e+07,0.133969,34309.439107,0,0
4,2026-05-20,(2013 TT5),0.032662,0.014607,26.30,2.652472e+07,0.177307,13356.758942,0,0


In [4]:
from sklearn.svm import SVC 
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split , GridSearchCV

In [5]:
X = df.drop(["close_date","astreoid_name","min_distance_km","potentially_hazardous"],axis=1)  
Y = df["potentially_hazardous"] 

In [6]:
x_train , x_test , y_train , y_test = train_test_split(X,Y,test_size=0.3,random_state=22) 

In [7]:
sc = StandardScaler() 
x_train_sc = sc.fit_transform(x_train) 
x_test_sc = sc.transform(x_test) 

In [8]:
"""
! Some of the errors here were resolved using assistance from Google Gemini.
"""
parameters = {
    "kernel":("rbf","linear","poly","sigmoid"),
    "gamma":["scale", "auto", 0.01, 0.1, 1],
    "C":[1,10,100,1000]  
}

In [9]:
svc_model = SVC(probability=True,random_state=22) 

model = GridSearchCV(estimator=svc_model,param_grid=parameters,cv=5) 
model.fit(x_train_sc,y_train) 
model.best_params_

{'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}

In [10]:
predict = model.predict(x_test_sc) 

In [11]:
print(classification_report(y_test,predict))

              precision    recall  f1-score   support

           0       0.91      0.83      0.87        98
           1       0.32      0.50      0.39        16

    accuracy                           0.78       114
   macro avg       0.62      0.66      0.63       114
weighted avg       0.83      0.78      0.80       114



In [12]:
print(model.score(x_train_sc,y_train)) 
print(model.score(x_test_sc,y_test))

0.8383458646616542
0.7807017543859649
